## Tokenization

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request


from pathlib import Path

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# URL for the raw tiny shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

path = Path('tiny-shakespeare.txt')

# Download the file if it doesn't already exist
if not path.exists():
    print("Downloading tiny-shakespeare.txt...")
    urllib.request.urlretrieve(url, path)
    print("Download complete!")

# Now this will work smoothly
text = path.read_text(encoding='utf-8')
print(f"Success! Loaded {len(text):,} characters.")

Success! Loaded 1,115,394 characters.


In [9]:
print(text[:300])  # Print the first 1000 characters of the text

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## Building a character level tokenizer

In [10]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.chars = sorted(list(set(vocabulary)))
        self.vocab_size = len(self.chars)
        self.token_id_for_char = {char: token_id for token_id, char in enumerate(self.chars)}
        self.char_for_token_id = {token_id: char for token_id, char in enumerate(self.chars)}


    @staticmethod
    def train_from_text(text):
        return CharTokenizer(text)

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return ''.join(chars)

    def vocabulary_size(self):
        return self.vocab_size

In [11]:
tokenizer = CharTokenizer.train_from_text(text)

In [12]:
tokenizer

In [13]:
encoding = tokenizer.encode("Hello World")

print(encoding)

tensor([20, 43, 50, 50, 53,  1, 35, 53, 56, 50, 42])


In [14]:
print(tokenizer.decode(encoding))

Hello World


In [15]:
tokenizer.vocabulary_size()

65

## Implementing a Dataset

In [1]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        if self.transform:
            item = self.transform(item)
        return item

# Create DataLoader
def create_data_loader(data, transform, batch_size):
    dataset = CustomDataset(data, transform)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


## TokenIdDataset

In [16]:
class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        # TODO: Save data and block size
        self.data = data
        self.block_size = block_size

    def __len__(self):
        # TODO: If every position can be a start of an item,
        # and all items should be "block_size", compute the size
        # of the dataset
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        # TODO: Check if the input position is valid
        if pos < 0 or pos >= len(self):
            raise IndexError("Invalid position")
        assert pos < len(self.data) - self.block_size, "Position out of range for block size"

        # TODO: Get an item from position "pos"
        x = self.data[pos:pos + self.block_size]
        # TODO: Get a target item (shifted by one position)
        y = self.data[pos + 1:pos + self.block_size + 1]

        # TODO: Return both
        return x, y

In [17]:
# Step 2 - Tokenize the Text

# TODO: Encode text using the tokenizer
# Create "TokenIdsDataset" with the tokenized text, and block_size=64
tokenized_text = tokenizer.encode(text)
dataset = TokenIdsDataset(tokenized_text, block_size=64)

In [18]:
tokenized_text

tensor([18, 47, 56,  ..., 45,  8,  0])

In [19]:
dataset

In [20]:
x, y = next(iter(dataset))

In [21]:
x

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50])

In [22]:
y

tensor([47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53,
        56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,  1,
        44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1, 57,
        54, 43, 39, 49,  8,  0,  0, 13, 50, 50])

In [23]:
x.shape

torch.Size([64])

In [25]:
tokenizer.decode(x)

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAl'

In [24]:
y.shape

torch.Size([64])

In [26]:
tokenizer.decode(y)

'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll'

In [ ]:
# Step 3 - Retrieve the First Item from the Dataset

# TODO: Get the first item from the dataset
# Decode "x" using tokenizer.decode

In [36]:
# Step 4 - Use a DataLoader

# TODO: Get a single batch from the "dataloader"
# For this call the `iter` function, and pass DataLoader instance to it. This will create an iterator
# Then call the `next` function and pass the iterator to it to get the first training batch

In [27]:
from torch.utils.data import DataLoader, RandomSampler

# RandomSampler allows to read random items from a datasset
sampler = RandomSampler(dataset, replacement=True)
# Dataloader will laod two random samplers using the sampler
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)

In [28]:
sampler

In [29]:
dataloader

In [30]:
x, y = next(iter(dataloader))

In [37]:
# TODO: Decode input item

In [31]:
x

tensor([[ 1, 44, 47, 50, 50,  1, 51, 63,  1, 40, 43, 50, 50, 63, 11,  1, 53, 52,
         43,  1, 44, 56, 59, 47, 58, 44, 59, 50,  1, 51, 43, 39, 50,  1, 61, 53,
         59, 50, 42,  1, 57, 43, 58,  0, 51, 43,  1, 58, 53,  1,  5, 58,  8,  1,
         14, 59, 58,  1, 58, 46, 43, 63,  1, 57],
        [14, 30, 13, 37, 10,  0, 13, 52, 42,  1, 21,  6,  1, 58, 53,  1, 49, 43,
         43, 54,  1, 39, 50, 50,  1, 58, 46, 47, 57,  8,  0,  0, 20, 17, 26, 30,
         37,  1, 14, 27, 24, 21, 26, 19, 14, 30, 27, 23, 17, 10,  0, 26, 53, 56,
         44, 53, 50, 49,  6,  1, 57, 53,  1, 44]])

In [33]:
tokenizer.decode(x[0])

" fill my belly; one fruitful meal would set\nme to 't. But they s"

In [34]:
tokenizer.decode(x[1])

'BRAY:\nAnd I, to keep all this.\n\nHENRY BOLINGBROKE:\nNorfolk, so f'

In [38]:
# TODO: Decode target item

In [39]:
tokenizer.decode(y[0])

"fill my belly; one fruitful meal would set\nme to 't. But they sa"

In [40]:
tokenizer.decode(y[1])

'RAY:\nAnd I, to keep all this.\n\nHENRY BOLINGBROKE:\nNorfolk, so fa'

## Embeddings

In [41]:
vocab_size = 10000
embedding_dim = 75
embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = embedding_dim)

In [42]:
embedding

Embedding(10000, 75)

In [44]:
token_ids = torch.LongTensor([11, 9783, 376, 45])
token_ids

tensor([  11, 9783,  376,   45])

In [45]:
embedded = embedding(token_ids)
embedded

tensor([[ 4.6879e-01, -3.3676e-01,  7.4276e-01, -3.3705e-01, -1.8083e+00,
         -9.4316e-01, -1.2998e+00,  6.5904e-01,  4.6504e-01,  1.1845e+00,
          1.1308e+00, -1.1316e+00,  1.2244e+00, -9.2449e-01, -1.0872e+00,
          1.3870e+00,  9.1985e-01, -7.0966e-01,  5.0751e-01,  1.8859e-01,
          1.2025e-01,  8.9369e-01,  1.6886e+00, -9.3812e-01,  2.3823e-01,
          5.2087e-01,  1.0631e-01, -4.0677e-01, -1.4321e+00, -5.9498e-01,
         -2.4957e-01,  6.5999e-01, -1.0632e+00, -7.3286e-01, -7.3016e-01,
         -5.1448e-01,  1.0371e+00, -4.1304e-01,  5.1760e-01, -1.2049e+00,
          1.6250e-01, -5.1314e-02, -2.5607e-01,  1.3675e+00,  1.4774e-01,
          3.0090e-01, -9.2086e-01, -7.8977e-01,  9.5793e-01,  7.6020e-01,
         -4.8191e-01,  1.4293e-01,  1.4700e-01,  1.3668e+00, -4.6855e-01,
         -7.1142e-01,  3.2796e-01,  5.5972e-02,  5.3776e-01,  8.4696e-01,
         -2.0883e-01,  2.5119e-01, -6.2796e-01,  7.7102e-01, -6.6091e-01,
          6.7940e-01, -6.9234e-01, -3.

In [46]:
embedded.shape

torch.Size([4, 75])

## Attention Mechanism

In [47]:
config = {
    "vocab_size": tokenizer.vocabulary_size(),
    "context_size": 256,
    "embedding_dim": 768,
    "num_attention_heads": 12,
    "number_of_layers": 10,
    "dropout_rate": 0.1,
    "activation_function": "gelu",
    "use_bias": False
}

config["head_size"] = config["embedding_dim"] // config["num_attention_heads"]

In [48]:
config

{'vocab_size': 65,
 'context_size': 256,
 'embedding_dim': 768,
 'num_attention_heads': 12,
 'number_of_layers': 10,
 'dropout_rate': 0.1,
 'activation_function': 'gelu',
 'use_bias': False,
 'head_size': 64}

In [49]:
class AttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding_dim = config["embedding_dim"]
        self.head_size = config["head_size"]
        self.num_heads = config["num_attention_heads"]
        self.use_bias = config["use_bias"]

        self.query_weight = nn.Linear(config["embedding_dim"], config["head_size"], config["use_bias"])
        self.key_weight = nn.Linear(config["embedding_dim"], config["head_size"], config["use_bias"])
        self.value_weight = nn.Linear(config["embedding_dim"], config["head_size"], config["use_bias"])

        # randomly initialized dropout layer to prevent overfitting
        self.dropout = nn.Dropout(config["dropout_rate"])

        # helps to prevent the model from attending to future tokens in the sequence
        casual_attention_mask = torch.tril(torch.ones(config["context_size"], config["context_size"])).unsqueeze(0).unsqueeze(0)
        # registers the casual attention mask as a buffer, which means 
        # it will be saved and loaded with the model, but it won't be updated during training i.e it is not a learnable parameter
        self.register_buffer("casual_attention_mask", casual_attention_mask)

    def forward(self, input_tensor):
        # input_tensor shape: (batch_size, seq_len/context window i.e context_size, embedding_dim)
        # Compute queries, keys, and values
        batch_size, token_num, embedding_dim = input_tensor.shape
        Q = self.query_weight(input_tensor)  # (batch_size, seq_len, head_size)
        K = self.key_weight(input_tensor)    # (batch_size, seq_len, head_size)
        V = self.value_weight(input_tensor)  # (batch_size, seq_len, head_size)

        # Compute attention scores
        attention_scores = torch.matmul(Q, K.transpose(1, 2))   # (batch_size, seq_len, seq_len)
        attention_scores = attention_scores.masked_fill(self.casual_attention_mask[:token_num, :token_num] == 0, -torch.inf)  # Apply casual mask
        attention_scores = attention_scores / (K.shape[-1] ** 0.5)  # Scale the scores so that they don't grow too large with the dimension of the key vectors
        attention_scores = torch.softmax(attention_scores, dim=-1)  # (batch_size, seq_len, seq_len)
        # dim=-1 means we are applying softmax across the last dimension, which is the sequence length dimension.
        # This ensures that for each query, the attention weights across all keys sum to 1.

        attention_scores = self.dropout(attention_scores)  # Apply dropout to the attention scores

        # Compute the output as a weighted sum of values
        output = torch.matmul(attention_scores, V)  # (batch_size, seq_len, head_size)
        return output

## Random Input Tensor

In [50]:
input = torch.rand(8, config["context_size"], config["embedding_dim"])
input

tensor([[[0.2981, 0.5798, 0.7918,  ..., 0.6880, 0.4926, 0.3758],
         [0.3903, 0.2452, 0.1385,  ..., 0.1164, 0.0635, 0.7208],
         [0.8366, 0.3579, 0.7697,  ..., 0.8973, 0.0904, 0.3497],
         ...,
         [0.5767, 0.9607, 0.3217,  ..., 0.4913, 0.5269, 0.5709],
         [0.4264, 0.9248, 0.4526,  ..., 0.7857, 0.5019, 0.1273],
         [0.3779, 0.3075, 0.2007,  ..., 0.3342, 0.3962, 0.9610]],

        [[0.5874, 0.0297, 0.8866,  ..., 0.2147, 0.9553, 0.0439],
         [0.6994, 0.5014, 0.7478,  ..., 0.0200, 0.5832, 0.2569],
         [0.5633, 0.2471, 0.7284,  ..., 0.1502, 0.9182, 0.2001],
         ...,
         [0.3179, 0.5560, 0.9521,  ..., 0.9716, 0.5984, 0.5083],
         [0.5292, 0.3527, 0.2255,  ..., 0.5570, 0.1897, 0.6084],
         [0.6466, 0.8793, 0.0594,  ..., 0.6373, 0.5173, 0.1825]],

        [[0.1700, 0.1781, 0.4691,  ..., 0.2817, 0.8215, 0.5073],
         [0.6867, 0.9265, 0.0136,  ..., 0.0276, 0.0220, 0.4927],
         [0.9678, 0.7606, 0.1301,  ..., 0.9722, 0.3918, 0.

In [51]:
input.shape

torch.Size([8, 256, 768])

In [52]:
input.size()

torch.Size([8, 256, 768])

In [53]:
ah = AttentionHead(config)
ah

AttentionHead(
  (query_weight): Linear(in_features=768, out_features=64, bias=False)
  (key_weight): Linear(in_features=768, out_features=64, bias=False)
  (value_weight): Linear(in_features=768, out_features=64, bias=False)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [54]:
output = ah(input)
output

tensor([[[[-1.7321e-01,  3.5905e-01,  2.3283e-01,  ..., -7.2565e-01,
            6.1075e-03,  5.9940e-01],
          [-3.1299e-01,  2.0091e-01,  2.0397e-02,  ..., -6.0228e-01,
           -6.0785e-02,  5.5377e-01],
          [-3.1086e-01,  1.8670e-01,  5.8599e-02,  ..., -5.0129e-01,
            5.2966e-02,  5.7215e-01],
          ...,
          [-2.2355e-01, -1.8491e-02, -1.0922e-02,  ..., -2.6987e-01,
            3.3255e-02,  4.1605e-01],
          [-2.2862e-01, -2.0883e-02, -3.7891e-03,  ..., -2.8834e-01,
            3.9467e-02,  4.2681e-01],
          [-2.4004e-01, -2.6827e-02, -1.1244e-02,  ..., -2.8550e-01,
            4.0398e-02,  4.4092e-01]],

         [[-3.9505e-01,  2.1763e-01, -3.3271e-01,  ..., -3.4684e-01,
           -3.3490e-01,  2.1101e-01],
          [-3.3014e-01,  3.5107e-01, -2.3645e-01,  ..., -2.9523e-01,
           -2.1147e-01,  2.3122e-01],
          [-3.4185e-01,  2.5324e-01, -9.9723e-02,  ..., -3.1423e-01,
           -1.0906e-01,  3.6857e-01],
          ...,
     

In [55]:
output.shape

torch.Size([1, 8, 256, 64])

## Multi Head Attention

In [56]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config["num_attention_heads"]
        self.embedding_dim = config["embedding_dim"]

        # Create a list of attention heads
        self.attention_heads = nn.ModuleList([AttentionHead(config) for _ in range(self.num_heads)])

        # Linear layer to combine the outputs of the attention heads
        self.output_linear = nn.Linear(self.embedding_dim, self.embedding_dim)
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input_tensor):
        # Compute the output of each attention head
        head_outputs = [head(input_tensor) for head in self.attention_heads]

        # Concatenate the outputs along the last dimension (head_size)
        concatenated_output = torch.cat(head_outputs, dim=-1)  # (batch_size, seq_len, num_heads * head_size)

        # Pass through the linear layer to get the final output
        output = self.output_linear(concatenated_output)  # (batch_size, seq_len, embedding_dim)
        dropout_output = self.dropout(output)  # Apply dropout to the final output
        return dropout_output

In [57]:
mha = MultiHeadAttention(config)
mha

MultiHeadAttention(
  (attention_heads): ModuleList(
    (0-11): 12 x AttentionHead(
      (query_weight): Linear(in_features=768, out_features=64, bias=False)
      (key_weight): Linear(in_features=768, out_features=64, bias=False)
      (value_weight): Linear(in_features=768, out_features=64, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (output_linear): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [58]:
output_mha = mha(input)
output_mha

tensor([[[[ 0.0840, -0.4789,  0.1423,  ..., -0.3449,  0.1819,  0.2146],
          [ 0.0281, -0.5469,  0.0929,  ..., -0.2497,  0.1787,  0.1051],
          [ 0.0468, -0.4758,  0.1791,  ..., -0.0000,  0.2917,  0.0603],
          ...,
          [ 0.0880, -0.5007,  0.0828,  ..., -0.3257,  0.1503,  0.1093],
          [ 0.0882, -0.5022,  0.0688,  ..., -0.0000,  0.0000,  0.0000],
          [ 0.0834, -0.5071,  0.0000,  ..., -0.3330,  0.1491,  0.0986]],

         [[ 0.2152, -0.4031, -0.1655,  ..., -0.1731, -0.0366,  0.1407],
          [ 0.1164, -0.5357,  0.1639,  ..., -0.0000,  0.0103,  0.1473],
          [ 0.0564, -0.4606,  0.1462,  ..., -0.2419,  0.0531,  0.1397],
          ...,
          [ 0.0798, -0.4989,  0.0659,  ..., -0.3236,  0.1529,  0.0000],
          [ 0.0769, -0.5011,  0.0697,  ..., -0.3262,  0.1552,  0.0915],
          [ 0.0816, -0.4957,  0.0550,  ..., -0.3155,  0.1580,  0.0957]],

         [[ 0.1453, -0.6719,  0.1016,  ..., -0.0000,  0.1491, -0.0387],
          [ 0.1485, -0.6708,  

In [59]:
output_mha.shape

torch.Size([1, 8, 256, 768])

In [2]:
images = load_images()

NameError: name 'load_images' is not defined

In [ ]:
images_dataset = CustomDataset(images, transform=resize_image)

dataset_len = len(images_dataset)

first_image = images_dataset[0]

for sample in range(len(images_dataset)):
    print(sample.shape)  # Print the shape of each image
    image = images_dataset[sample]
    break  # Just to check the first sample